# Teams Bot Functionality Test

Learn and test the Microsoft Teams integration:

- Teams activity models
- Adaptive Card builders currently implemented: response, HITL, error, welcome
- HITL approve/reject card payloads
- HMAC verification behavior
- Optional router import and webhook smoke test

Note: `docs/PROJECT_CONTEXT.md` mentions `build_thinking_card()`, but the current `src/teams/cards.py` does not define it. This notebook follows the actual code.

In [ ]:
from pathlib import Path
import json
import sys

cwd = Path.cwd().resolve()
project_root = next((p for p in [cwd, *cwd.parents] if (p / 'src' / 'teams').exists()), cwd)
src_path = project_root / 'src'
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

print('project_root:', project_root)

## 1. Teams Activity Models

In [ ]:
from teams.models import TeamsActivity, TeamsUser, TeamsConversation

message_payload = {
    'type': 'message',
    'text': 'Why did retention drop last month?',
    'from': {'id': 'user-1', 'name': 'Ava'},
    'conversation': {'id': 'conv-1', 'isGroup': False},
}
activity = TeamsActivity(**message_payload)
print(activity)
print('from id:', activity.from_.id)
assert activity.from_.id == 'user-1'
assert activity.conversation.id == 'conv-1'

## 2. Adaptive Card Builders

In [ ]:
from teams.cards import build_response_card, build_hitl_card, build_error_card, build_welcome_card

result = {
    'final_summary': 'Retention dropped due to churn in SMB.',
    'intent': 'metric_analysis',
    'confidence': 0.91,
    'anomalies': ['GRR below 85% threshold'],
    'auto_tickets': ['DATA-5010'],
}
response_card = build_response_card(result)
print(json.dumps(response_card, indent=2)[:2000])
assert response_card['type'] == 'message'
assert response_card['attachments'][0]['content']['type'] == 'AdaptiveCard'

hitl = {'message': 'Create tickets?', 'anomalies': ['GRR below threshold'], 'count': 1}
hitl_card = build_hitl_card(hitl, thread_id='conv-1', query='Why did retention drop?')
actions = hitl_card['attachments'][0]['content']['actions']
print('actions:', actions)
assert actions[0]['data']['action'] == 'approve_tickets'
assert actions[1]['data']['action'] == 'reject_tickets'

assert build_error_card('boom')['type'] == 'message'
assert build_welcome_card()['type'] == 'message'

## 3. Optional Bot Router Import

Importing `teams.bot` imports graph/api middleware too. If dependencies are missing, this cell reports that cleanly.

In [ ]:
bot_import_error = None
try:
    import teams.bot as bot
    print('router:', bot.router)
    print('hmac disabled passes:', bot._verify_hmac(b'body', None))
except Exception as exc:
    bot_import_error = exc
    print('teams.bot import failed:', repr(exc))

## 4. Optional Handler Smoke Tests

In [ ]:
RUN_HANDLER_TESTS = False

if RUN_HANDLER_TESTS and bot_import_error is None:
    import asyncio
    welcome = asyncio.run(bot._handle_conversation_update(TeamsActivity(**{
        'type': 'conversationUpdate',
        'from': {'id': 'bot'},
        'membersAdded': [{'id': 'user-1', 'name': 'Ava'}],
    })))
    print(json.dumps(welcome, indent=2)[:1000])
else:
    print('Skipped. Set RUN_HANDLER_TESTS = True after teams.bot imports cleanly.')